# Day 6 · 第一个线性回归模型

目标：会调用 fit/predict，理解参数、损失、基准和 Pipeline。
基础 50 分钟、深度 35 分钟、广度 25 分钟、作业 30 分钟；Bonus 30 分钟。

### 1. 模型如何表达关系

预测值：$\hat y=b_0+b_1x_1+\cdots+b_px_p$。
截距是所有特征为零时的模型值；系数是控制其他输入不变时，某特征每增加一单位的预测变化。
训练寻找使训练样本平方残差之和最小的系数。
线性表示“对参数线性”，并不意味着所有真实关系都线性。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "assignments/bootcamp/week1/data/raw/student-mat.csv").exists())
raw = pd.read_csv(ROOT / "assignments/bootcamp/week1/data/raw/student-mat.csv", sep=";")
# 固定划分，第三至七天始终使用相同学生；实际行数为 276/59/60。
train, heldout = train_test_split(raw, test_size=0.30, random_state=42)
val, test = train_test_split(heldout, test_size=0.50, random_state=42)
train, val, test = train.copy(), val.copy(), test.copy()
print("训练/验证/测试：", len(train), len(val), len(test))


### 2. 先训练一个单特征模型

X 必须是二维，所以 train[["G2"]] 用双层中括号。
fit 从训练样本学习参数；predict 使用参数生成结果；score 对 LinearRegression 默认是 R²。

In [ ]:
from sklearn.linear_model import LinearRegression
one = LinearRegression().fit(train[["G2"]], train["G3"])
print("截距：", one.intercept_, "斜率：", one.coef_[0])
print("G2=12 的预测：", one.predict(pd.DataFrame({"G2":[12]}))[0])

### 3. 多特征 Pipeline 与基准

SimpleImputer 只在 fit 输入上学习中位数，predict 自动用相同统计量转换。
原始 UCI 文件无缺失，但这个结构让后来加入的缺失处理不容易泄漏。
这里采用第二阶段成绩已知的教学场景，不使用可能包含未来信息的期末累计缺勤。

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
features = ["G1", "G2", "studytime", "failures"]
target = "G3"
X_train, y_train = train[features], train[target]
X_val, y_val = val[features], val[target]
def make_model():
    return Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("regressor", LinearRegression())])
def metrics(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "MSE": mean_squared_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred)}
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
model = make_model().fit(X_train, y_train)
val_pred = model.predict(X_val)
display(pd.DataFrame({
    "baseline": metrics(y_val, baseline.predict(X_val)),
    "linear": metrics(y_val, val_pred)}).T)


In [ ]:
coefficients = pd.Series(model.named_steps["regressor"].coef_, index=features)
display(coefficients)
print("截距：", model.named_steps["regressor"].intercept_)
print("填补中位数：", model.named_steps["imputer"].statistics_)

### 4. 深度：系数与共线性

G1/G2 相关，因此加入 G2 后 G1 系数可能变化甚至变号。
不能根据系数绝对值直接比较不同单位特征的“重要性”。
studytime 是有序档位，将它当数字假设相邻档位的效果相同；后面的 Bonus 可比较独热编码。
系数是统计关联，不能据此声称改变学习行为就一定改变成绩。
无约束线性模型可能预测出 0～20 以外的值，不能为美化测试结果临时裁剪。

In [ ]:
display(train[["G1","G2"]].corr())
print("验证集范围外预测数：", int(((val_pred < 0) | (val_pred > 20)).sum()))
for col, value in coefficients.items():
    print(f"其他输入固定，{col} 增加 1，模型预测改变 {value:.3f} 分。")

### 5. 广度：数值和类别特征一起处理

独热编码将类别转成指示列；handle_unknown="ignore" 允许预测时出现未见类别。
drop="first" 避免完整哑变量加截距带来的冗余。
预处理器也要在 Pipeline 内，每次 fit 都只看到训练数据。

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
mixed_features = features + ["school", "internet"]
preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore", drop="first"))
    ]), ["school", "internet"])
])
mixed_model = Pipeline([("prepare", preprocess), ("regressor", LinearRegression())])
mixed_model.fit(train[mixed_features], train["G3"])
print("类别扩展方案 validation MAE：",
      mean_absolute_error(val["G3"], mixed_model.predict(val[mixed_features])))

### DAY 6 ASSIGNMENT

基础：对比 G2 单特征与四特征方案的验证 MAE。
深度：解释一个系数以及为何不代表因果关系。
广度：对比加入 school/internet 的方案，明确这属于验证集实验。
Bonus：使用 StandardScaler + Ridge，并只在训练集内部做 5 折交叉验证比较 alpha=0.1/1/10。
验收：与均值基准比较，保存验证结果；测试集不参与选择。
第七天教学主线仅比较预先规定的两个线性方案；若纳入你自己的 Bonus 候选，必须先重写选择方案再做最终评估。

In [ ]:
# 作业区


### 参考答案与 Bonus 示例

In [ ]:
# 两个预先规定的候选方案；选择标准仅为验证集 MAE。
feature_sets = {"G2_only": ["G2"], "multi": ["G1","G2","studytime","failures"]}
candidates = {}
rows = []
for name, cols in feature_sets.items():
    fitted = make_model().fit(train[cols], train["G3"])
    candidates[name] = fitted
    rows.append({"model": name, **metrics(val["G3"], fitted.predict(val[cols]))})
validation_results = pd.DataFrame(rows).set_index("model")
display(validation_results)
chosen = validation_results["MAE"].idxmin()
chosen_features = feature_sets[chosen]
print("冻结方案：", chosen, chosen_features)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, KFold
cv = KFold(n_splits=5, shuffle=True, random_state=42)
for alpha in [0.1,1,10]:
    ridge = Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler()), ("ridge", Ridge(alpha=alpha))])
    losses = -cross_val_score(ridge, X_train, y_train, cv=cv,
                              scoring="neg_mean_absolute_error")
    print(alpha, "CV MAE mean/std:", losses.mean(), losses.std())

### 复盘（填写）

fit 和 predict 的区别：___。为什么填补也需要 fit：___。我的候选方案记录：___。